# 00 — Génération des données fictives

Ce notebook (illustratif, **hors CI**) reconstruit la chaîne de données fictive du projet,
de bout en bout, puis en donne un aperçu.

## Flux de données

```
templates/   gabarits Excel vides (format réel, versionnés)
   │  fill_fake_data.main()      → série fictive *_fictif.xlsx dans data/
   ▼
data/*_fictif.xlsx
   │  export_internes.exporter_csv()   (normalisation → format interne)
   ▼
data/aphp_data.csv · survival_data.csv · regional_data.csv
```

Les **3 CSV internes** produits :

| Fichier | Clé | Contenu |
|---|---|---|
| `aphp_data.csv` | `(annee, entite, appareil, organe)` | patients & séjours AP-HP / GHU |
| `survival_data.csv` | `+ (stade, population)` | survie à 1 an / 5 ans par stade |
| `regional_data.csv` | `(annee, entite, appareil, organe)` | comparaison par type d'établissement |

> ⚠ **Données entièrement simulées** (`seed 42`), à titre illustratif uniquement.
> Ne reflètent pas la réalité clinique.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
from pathlib import Path

# Chaîne actuelle : templates/ -> data/*.xlsx -> 3 CSV internes
from fill_fake_data import main as generer_fictif      # templates/ -> data/*.xlsx
from export_internes import exporter_csv               # xlsx -> 3 CSV internes

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## 1. Génération

`generer_fictif()` remplit les gabarits de `templates/` et écrit les `*_fictif.xlsx`
dans `data/`. `exporter_csv()` lit ces xlsx et écrit les 3 CSV internes.

In [2]:
generer_fictif()                                # templates/ -> data/*_fictif.xlsx
exporter_csv(dossier_data=str(DATA_DIR))        # data/*.xlsx -> data/*.csv (format interne)

print('\nCSV internes produits :')
for nom in ['aphp_data.csv', 'survival_data.csv', 'regional_data.csv']:
    df = pd.read_csv(DATA_DIR / nom)
    print(f'  {nom:<20} {len(df):>7,} lignes - {len(df.columns)} colonnes')

OECI : série annuelle 2019→2023 (base tirée une seule fois)…


  → 2019 (×1.000) : indicateurs_oeci_2019_M12_fictif.xlsx  [OK]


  → 2020 (×0.900) : indicateurs_oeci_2020_M12_fictif.xlsx  [OK]


  → 2021 (×0.945) : indicateurs_oeci_2021_M12_fictif.xlsx  [OK]


  → 2022 (×0.969) : indicateurs_oeci_2022_M12_fictif.xlsx  [OK]


...

  Age >= 18 ans: 86980 lignes remplies


  Total: 88447 lignes (clé absente des 2 âges : 0)


  → écrit : /Users/remi/GitHub/cancer_network_report/notebooks/../src/../data/canceroBR_16-25_Sej_13032026_fictif.xlsx

Terminé.
Export interne (fictif=True) : source ../data/ → ../data/


  → aphp_data.csv          2,385 lignes, 14 colonnes
  → survival_data.csv     28,280 lignes, 9 colonnes
  → regional_data.csv      3,950 lignes, 10 colonnes

CSV internes produits :
  aphp_data.csv          2,385 lignes - 14 colonnes
  survival_data.csv     28,280 lignes - 9 colonnes


  regional_data.csv      3,950 lignes - 10 colonnes


## 2. Aperçu AP-HP

Total AP-HP (toutes localisations) par année, puis distribution par appareil
pour la dernière année.

In [3]:
aphp_df = pd.read_csv(DATA_DIR / 'aphp_data.csv')

# AP-HP / TOTAL : une ligne par année
aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil == 'TOTAL')][
    ['annee', 'nb_patients', 'nb_nouveaux_patients',
     'nb_sejours_chirurgie', 'nb_sejours_chimiotherapie',
     'nb_sejours_radiotherapie', 'nb_sejours_palliatifs']
].sort_values('annee').reset_index(drop=True)

,annee,nb_patients,nb_nouveaux_patients,nb_sejours_chirurgie,nb_sejours_chimiotherapie,nb_sejours_radiotherapie,nb_sejours_palliatifs
0,2019,62875,31484,24385,27115,17251,6907
1,2020,56594,28333,21968,24421,15607,6373
2,2021,59444,29803,23130,25708,16403,6702
3,2022,60984,30623,23720,26371,16834,6832
4,2023,62500,31325,24275,26975,17180,6899


In [4]:
# Distribution par appareil - derniere annee
derniere_annee = int(aphp_df.annee.max())
dist = aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil != 'TOTAL')
               & (aphp_df.organe == 'TOTAL') & (aphp_df.annee == derniere_annee)]
print(f'Distribution par appareil - {derniere_annee}')
dist[['appareil', 'nb_patients']].sort_values('nb_patients', ascending=False).reset_index(drop=True)

Distribution par appareil - 2023


,appareil,nb_patients


## 3. Aperçu régional

Totaux par **type d'établissement** (colonne `entite` : AP-HP, Clinique, CH, CHU, PSPH, CLCC)
pour la dernière année disponible.

In [5]:
regional_df = pd.read_csv(DATA_DIR / 'regional_data.csv')

# Totaux par type d'etablissement (colonne `entite`) - derniere annee
reg_derniere = int(regional_df.annee.max())
reg = regional_df[(regional_df.appareil == 'TOTAL') & (regional_df.organe == 'TOTAL')
                  & (regional_df.annee == reg_derniere)]
print(f"Activite par type d'etablissement - {reg_derniere}")
reg[['entite', 'nb_patients', 'nb_nouveaux_patients',
     'nb_sejours_chirurgie', 'nb_sejours_chimiotherapie']
].sort_values('nb_patients', ascending=False).reset_index(drop=True)

Activite par type d'etablissement - 2025


,entite,nb_patients,nb_nouveaux_patients,nb_sejours_chirurgie,nb_sejours_chimiotherapie
0,AP-HP,35603,18822,19168,10562
1,Clinique,26819,14053,13665,7299
2,CH,21264,11252,11893,6509
3,PSPH,12805,6798,6032,3246
4,CLCC,10898,5708,4022,2133


## 4. Aperçu survie

Survie AP-HP par stade (`I-III`, `IV`) pour la **population `tous`**, dernière année.
On filtre `population == 'tous'` pour ne pas mélanger avec `nouveaux`.

In [6]:
survival_df = pd.read_csv(DATA_DIR / 'survival_data.csv')

# Survie AP-HP par stade, population « tous », derniere annee, niveau appareil agrege (organe=TOTAL)
surv_derniere = int(survival_df.annee.max())
surv = survival_df[(survival_df.entite == 'AP-HP') & (survival_df.population == 'tous')
                   & (survival_df.organe == 'TOTAL') & (survival_df.appareil == 'TOTAL')
                   & (survival_df.annee == surv_derniere)]
print(f'Survie AP-HP par stade (population tous) - {surv_derniere}')
surv[['stade', 'nb_patients_stade', 'survie_1an', 'survie_5ans']
].sort_values('stade').reset_index(drop=True)

Survie AP-HP par stade (population tous) - 2023


,stade,nb_patients_stade,survie_1an,survie_5ans
0,I-III,18946,83.8,73.8
1,IV,4319,38.9,25.0


## 5. Validation Plotly

Contrôle visuel rapide : évolution du nombre total de patients AP-HP.

In [7]:
import plotly.express as px

d = aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil == 'TOTAL')].sort_values('annee')
fig = px.bar(d, x='annee', y='nb_patients',
             title='AP-HP - Total patients par annee (donnees fictives)',
             labels={'annee': 'Annee', 'nb_patients': 'Patients'},
             color_discrete_sequence=['#003189'])
fig.show()